## Libraries import

In [1]:
import json
import os
import numpy as np
from glob import glob
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
print("DONE")

2026-06-16 07:17:36.471805: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781594256.701327      16 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781594256.764953      16 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1781594257.294409      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781594257.294447      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781594257.294449      16 computation_placer.cc:177] computation placer alr

DONE


## Carica vocabolario

In [2]:
# Modifica il percorso se necessario
vocab_path = "/kaggle/input/datasets/umerellous/primus/vocabulary.json"

with open(vocab_path, "r") as f:
    vocab = json.load(f)

idx2token = {idx: token for token, idx in vocab.items()}
vocab_size = len(vocab)
pad_idx = vocab.get("<PAD>", 0)
unk_idx = vocab.get("<UNK>", 0)

print(f"Vocabolario: {vocab_size} token")
print(f"<PAD> idx: {pad_idx}, <UNK> idx: {unk_idx}")

Vocabolario: 1785 token
<PAD> idx: 0, <UNK> idx: 3


## Leggi file .semantic

In [3]:
print("START")

semantic_root = "/kaggle/input/datasets/umerellous/primus/primus/"
sequences_tokens = []  # lista di liste di stringhe

i = 0
for filepath in glob(os.path.join(semantic_root, "**/*.semantic"), recursive=True):
    with open(filepath, "r") as f:
        line = f.read().strip()
        if not line:
            continue
        tokens = line.split()  # separa per spazi o tab
        if len(tokens) > 1:
            sequences_tokens.append(tokens)
    i+=1


print(f"Trovate {len(sequences_tokens)} sequenze di token")

START
Trovate 87678 sequenze di token


## Converti token in indici (con gestione UNK)

In [4]:
def tokens_to_indices(tokens, vocab, unk_idx):
    return [vocab.get(tok, unk_idx) for tok in tokens]

sequences_idx = [tokens_to_indices(seq, vocab, unk_idx) for seq in sequences_tokens]

# Statistiche di lunghezza
lengths = [len(seq) for seq in sequences_idx]
print(f"Lunghezza media: {np.mean(lengths):.1f}")
print(f"Lunghezza max: {max(lengths)}")
print(f"Lunghezza min: {min(lengths)}")

Lunghezza media: 23.9
Lunghezza max: 58
Lunghezza min: 3


## Crea finestre (input=seq_len, target=nota successiva)


In [5]:
seq_len = 20   # GIUSTIFICA
X, y = [], []

for seq in sequences_idx:
    if len(seq) < seq_len + 1:
        continue
    for i in range(len(seq) - seq_len):
        X.append(seq[i:i+seq_len])
        y.append(seq[i+seq_len])

X = np.array(X, dtype=np.int32)
y = np.array(y, dtype=np.int32)
print(f"Shape X: {X.shape}, shape y: {y.shape}")

Shape X: (413230, 20), shape y: (413230,)


## Split train/validation

In [6]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.1, random_state=42)
print(f"Train: {X_train.shape}, Val: {X_val.shape}")

Train: (371907, 20), Val: (41323, 20)


## Costruisci modello LSTM bidirezionale

In [7]:
import tensorflow as tf

# Forza CPU (per aggirare errore CUDA)
tf.config.set_visible_devices([], 'GPU')

embedding_dim = 128
lstm_units = 256
dropout_rate = 0.2

model = Sequential([
    Embedding(vocab_size, embedding_dim, input_shape=(seq_len,)),  # input_shape invece di input_length
    Bidirectional(LSTM(lstm_units, dropout=dropout_rate, return_sequences=False)),
    Dense(vocab_size, activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.build(input_shape=(None, seq_len))  # build esplicita
model.summary()

2026-06-16 07:44:14.584712: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:103: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 20, 128)        │       228,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 512)            │       788,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1785)           │       915,705 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,932,665 (7.37 MB)

 Trainable params: 1,932,665 (7.37 MB)

 Non-trainable params: 0 (0.00 B)

## Addestramento modello

In [8]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    batch_size=64,
    epochs=40,
    verbose=1
)

Epoch 1/40
5812/5812 ━━━━━━━━━━━━━━━━━━━━ 594s 102ms/step - accuracy: 0.3149 - loss: 2.8611 - val_accuracy: 0.3814 - val_loss: 2.4024
Epoch 2/40
5812/5812 ━━━━━━━━━━━━━━━━━━━━ 607s 104ms/step - accuracy: 0.3997 - loss: 2.2930 - val_accuracy: 0.4216 - val_loss: 2.2223
Epoch 3/40
5812/5812 ━━━━━━━━━━━━━━━━━━━━ 597s 103ms/step - accuracy: 0.4353 - loss: 2.1141 - val_accuracy: 0.4403 - val_loss: 2.1397
Epoch 4/40
5812/5812 ━━━━━━━━━━━━━━━━━━━━ 599s 103ms/step - accuracy: 0.4631 - loss: 1.9848 - val_accuracy: 0.4510 - val_loss: 2.1075
Epoch 5/40
5812/5812 ━━━━━━━━━━━━━━━━━━━━ 593s 102ms/step - accuracy: 0.4864 - loss: 1.8774 - val_accuracy: 0.4563 - val_loss: 2.0962
Epoch 6/40
5812/5812 ━━━━━━━━━━━━━━━━━━━━ 589s 101ms/step - accuracy: 0.5071 - loss: 1.7852 - val_accuracy: 0.4626 - val_loss: 2.0979
Epoch 7/40
5812/5812 ━━━━━━━━━━━━━━━━━━━━ 613s 100ms/step - accuracy: 0.5260 - loss: 1.7050 - val_accuracy: 0.4644 - val_loss: 2.1107
Epoch 8/40
5812/5812 ━━━━━━━━━━━━━━━━━━━━ 590s 102ms/step - ac

## Funzione di predizione (stile T9 musicale)

In [9]:
def predict_next_tokens(seed_tokens, model, vocab, idx2token, seq_len, top_k=3):
    """
    seed_tokens : lista di token (stringhe) di lunghezza qualsiasi
    Restituisce lista di (token, probabilità) per i top_k
    """
    # Converti seed in indici
    unk_idx = vocab.get("<UNK>", 0)
    seed_idx = [vocab.get(tok, unk_idx) for tok in seed_tokens]
    
    # Se la sequenza è più corta di seq_len, padding a sinistra con <PAD>
    if len(seed_idx) < seq_len:
        pad_idx = vocab.get("<PAD>", 0)
        seed_idx = [pad_idx] * (seq_len - len(seed_idx)) + seed_idx
    else:
        seed_idx = seed_idx[-seq_len:]
    
    input_batch = np.array([seed_idx], dtype=np.int32)
    probs = model.predict(input_batch, verbose=0)[0]
    
    # Ottieni i top_k indici
    top_indices = np.argsort(probs)[-top_k:][::-1]
    result = [(idx2token[i], float(probs[i])) for i in top_indices]
    return result

#Esempio di utilizzo (dopo il training)
seed = [
    "clef-G2", 
    "timeSignature-4/4", 
    "note-B4_quarter", 
    "note-B4_quarter", 
    "note-C5_quarter", 
    "note-D5_quarter", 
    "barline", 
    "note-D5_quarter", 
    "note-C5_quarter", 
    "note-B4_quarter", 
    "note-A4_quarter", 
    "barline", 
    "note-G4_quarter"
]

print(predict_next_tokens(seed, model, vocab, idx2token, seq_len, top_k=3))

[('note-D5_quarter', 0.2706553637981415), ('note-G4_quarter', 0.19337613880634308), ('rest-quarter', 0.148899108171463)]


## Salvataggio modello

In [10]:
model.save("/kaggle/working/lstm_music_model.h5")
print("Modello salvato in /kaggle/working/lstm_music_model.h5")

model.save("/kaggle/working/lstm_music_model.keras")
print("Modello salvato in /kaggle/working/lstm_music_model.keras")

Modello salvato in /kaggle/working/lstm_music_model.h5
Modello salvato in /kaggle/working/lstm_music_model.keras
